In [ ]:
# Standard library imports

import importlib
import sys
from pathlib import Path

# Project path setup

PROJECT_ROOT = Path("..").resolve()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Third-party imports

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from dotenv import load_dotenv
from scipy.stats import pearsonr, spearmanr

# Local project imports

from src import (
    config,
    data,
    evaluate,
    llm,
    prompts,
    qa,
    rag,
    scoring,
)

from src.notebook_eval import (
    essay_correlation,
    essay_evaluation,
    essay_plots,
    article_recitation_plots,
    judge_correlation
)

# Environment configuration

load_dotenv(override=True)

pd.set_option("display.max_columns", None)

# Data loading

DATA_PATH = Path("data/gpbam.json")
gpbam_df = pd.read_json(DATA_PATH)

X_LIMITS = {
    "quality_vs_answer_length": (0, 22),
    "quality_vs_generation_cost": (0, 9.5),
    "quality_vs_legal_reference_similarity": (0, 26),
}
Y_LIMITS = (0, 50)


In [ ]:
def ew_withrag(
    judge_instruction_name: str = "ji2",
    *,
    models=None,
    judge_models=None,
    device: str | None = None,
    allow_overwrite_existing_models: bool = False,
    results_path: str | Path | None = None,
    gpbam_path: str | Path = "data/gpbam.json",
    batch_size: int = 5,
    max_tokens=None,
    law_ret=None,
    require_confirmation: bool = True,
    verbose: bool = True,
    reload_modules=True,
):
    """
    Run the law-RAG essay-writing evaluation end to end: for each model, retrieve
    relevant German-law context and generate an answer to every gpbam task, then
    score it with the judge ensemble. Notebook version of
    essay_writing_lawrag_overnight.py.

    Answers ARE generated here (retrieval-augmented) —
    Pass a pre-built `law_ret` to skip rebuilding the
    retriever on repeat calls.

    Args:
        judge_instruction_name: Key into `prompts.JUDGE_INSTRUCTIONS_BY_NAME` selecting
            the judge instruction actually sent to the judge (e.g. "ji1", "ji2"). Also
            names the default `results_path` folder/file.
        models: Iterable of model names to generate answers with and judge. Defaults to
            `config.MODELS_DEV`. Deduplicated while preserving order.
        judge_models: Model names forming the judge ensemble. Defaults to
            `config.JUDGE_MODELS_DEV`.
        device: Torch device for the retriever's embedding model (e.g. "cuda:3", "cpu").
            If None, uses "cuda:3" when CUDA is available, else "cpu". Ignored when a
            pre-built `law_ret` is passed.
        allow_overwrite_existing_models: Controls what happens to a model already present
            in `results_path` (generated + judged in a prior run) — True regenerates and
            re-judges it, replacing its old rows; False (default) skips it, so a re-run
            resumes an interrupted run / adds new models without redoing (and re-paying
            for) generation + judging already done.
        results_path: CSV that is both read (resume cache) and written. If None, defaults
            to a with-RAG path derived from `judge_instruction_name`. Saved after every
            model so interrupted runs can resume.
        gpbam_path: Path to the gpbam JSON providing the task `facts` (prompts to answer)
            and `solutions` (reference answers the judge scores against).
        batch_size: Embedding batch size used only when the law knowledge base has to be
            built (i.e. the retriever isn't already fitted).
        max_tokens: Max tokens for each model's answer generation. None uses the
            generator's default.
        law_ret: Optional pre-built `rag.LawRetriever` to reuse across calls. Reusing one
            avoids re-loading the embedding model onto `device`, reconnecting to the DB,
            and rebuilding the query-rewrite cache. If None, a fresh retriever is created —
            but the knowledge base itself is persisted on disk and reopened from disk in
            either case, so passing None never rebuilds the KB unless the on-disk
            `documents` table (at the retriever's `kb_name`) is missing.
        require_confirmation: If True, print a path/instruction summary and gate the run
            behind an interactive 'yes' prompt. Set False to run non-interactively.
        verbose: If True, print retriever/per-model progress while running.

    Returns:
        The results DataFrame (also written to `results_path`).
    """

    load_dotenv(override=True)

    import torch

    if reload_modules:
        for mod in (scoring, qa, llm, prompts, config, rag, data, evaluate):
            importlib.reload(mod)

    # --- defaults that depend on other args / modules ---
    if models is None:
        models = config.MODELS_DEV
    if judge_models is None:
        judge_models = config.JUDGE_MODELS_DEV
    if device is None:
        device = "cuda:0" if torch.cuda.is_available() else "cpu"
    if results_path is None:
        results_path = (
            f"./zubaers_result/essay_writing/with_rag/"
            f"{judge_instruction_name}/with_rag_{judge_instruction_name}_result.csv"
        )
    results_path = Path(results_path)

    judge_instruction = prompts.JUDGE_INSTRUCTIONS_BY_NAME[judge_instruction_name]

    # --- data + retriever setup ---
    gpbam_df = pd.read_json(gpbam_path)

    data.ensure_law_data()
    laws = data.parse_german_laws()
    if verbose:
        print(f"Parsed {len(laws)} laws.")

    if law_ret is None:
        rewriter = qa.ReWriter()
        law_ret = rag.LawRetriever(rewriter=rewriter, verbose=1, device=device)

    if law_ret.table_ is None:
        print("Knowledge base not found. Building it now...")
        law_ret.fit(laws, batch_size=batch_size)
    else:
        print(f"Using existing knowledge base with {len(law_ret.table_)} chunks.")

    # --- judge ensemble ---

    judges = [
        scoring.Judge(model=model, prompt=prompts.build_judge_user(judge_instruction))
        for model in judge_models
    ]
    judge = scoring.JudgeEnsemble(judges, verbose=True)

    # --- safety checkpoint ---
    print("\n" + "=" * 80)
    print("RESULT CSV PATH CHECK")
    print("=" * 80)
    print(f"Results path (read and write):\n{results_path}")
    print("=" * 80)
    print(f"Judge instruction ({judge_instruction_name})")
    print(judge_instruction[:])
    print("=" * 80)
    print(f"Allow overwrite existing model rows: {allow_overwrite_existing_models}")
    print("\nBefore continuing, verify:")
    print("1. This is the correct existing CSV for this RAG run.")
    print("2. The CSV filename is correct.")
    print("3. The instruction text printed above matches judge_instruction_name "
          "(this is what's actually sent — ignore prompts.py's own JUDGE_USER print, "
          "that's an unrelated default and is not used here).")
    print("=" * 80)

    if require_confirmation:
        confirmation = input("Type 'yes' to continue running the script: ").strip().lower()
        if confirmation != "yes":
            raise RuntimeError(
                "Execution stopped. Please set `results_path` correctly before running again."
            )

    # --- run evaluation ---
    results_path.parent.mkdir(parents=True, exist_ok=True)

    if results_path.exists():
        print(f"Loading existing RAG results from {results_path}")
        model_study_df = pd.read_csv(results_path)
    else:
        print("No existing results found. Starting a fresh results DataFrame.")
        model_study_df = pd.DataFrame()

    completed_models = (
        set(model_study_df["model"].dropna()) if "model" in model_study_df else set()
    )

    for model in dict.fromkeys(models):
        if model in completed_models and not allow_overwrite_existing_models:
            print(f"Skipping completed model: {model}")
            continue

        if model in completed_models:
            print(f"Overwriting completed model: {model}")
            model_study_df = model_study_df[model_study_df["model"] != model].copy()

        rag_model = qa.AnswerGenerator(retriever=law_ret, model=model, max_tokens=max_tokens)
        model_df = evaluate.evaluate_model(
            rag_model,
            gpbam_df.facts.values,
            gpbam_df.solutions.values,
            judge=judge,
            verbose=verbose,
            model_name=model,
        )
        model_study_df = pd.concat([model_study_df, model_df], ignore_index=True)
        model_study_df.to_csv(results_path, index=False)
        completed_models.add(model)

    return model_study_df


In [ ]:
model_study_df = ew_withrag(judge_instruction_name="ji2")

In [ ]:
try:
    model_study_df
except NameError:
    model_study_df = pd.read_csv("/root/work/p25-plexam/experiments/zubaers_result/essay_writing/with_rag/ji2/with_rag_ji2_result.csv")

model_study_df.head(2)

In [ ]:
# Add legal reference similarity scores only if the column is absent

if "legal_ref_sim" not in model_study_df.columns:
    solutions = gpbam_df["solutions"].values
    legal_ref_scores = []

    for _, row in model_study_df.iterrows():
        answer = row["answer"]
        solution_index = int(row["index"])
        solution = solutions[solution_index]

        score = scoring.legal_ref_similarity(answer, solution)
        legal_ref_scores.append(score)

    model_study_df["legal_ref_sim"] = legal_ref_scores
else:
    print("Column 'legal_ref_sim' already exists in the DataFrame. Skipping computation.")

In [ ]:
# MAIN: score + legal_ref_sim
main = essay_evaluation.make_summary(df=model_study_df,
                                     caption=r"\textbf{Essay writing performance (with law retrieval).} Each model receives the facts of one of the 81 German public-law state-exam cases in GPBam and writes a complete legal essay (\emph{Gutachten}) in German, this time augmented with retrieved law: a rewriter LLM condenses the case facts into 3--5 standalone legal search queries, and the five best-matching statute passages, found by hybrid dense-plus-lexical search over a knowledge base built from the German federal statute corpus (\emph{gesetze-im-internet}), are prepended to the prompt as cited sources. Every essay is graded against the official reference solution by an ensemble of three LLM judges (gpt-5-nano, qwen3.6-35b-a3b, DeepSeek-V4-Flash), each awarding 0.0--1.0 for correctness, completeness and legal reasoning; an essay's score is the median over the three judges. \textbf{Score} is the mean of those medians over the 81 cases, as a percentage, with its bootstrap standard error ($_{\pm\mathrm{SE}}$, indicating how precisely the mean is estimated). \textbf{Legal Ref. Sim.} is the Jaccard overlap between the statutory references (law book and section, e.g.\ ``\S~42 VwGO'') cited in the essay and those cited in the reference solution, likewise a mean over the 81 cases with bootstrap SE. Higher is better for both. Models are ordered by Score (ascending)."
,
                                     label="tab:model_comparison_with_rag")
main

In [ ]:
res_main = essay_correlation.correlate(
    main,
    recitation_csv="zubaers_result/article_recitation/article_recitation.csv",
)
res_main

In [ ]:
essay_correlation.to_latex(
    res_main,
    label="tab:essay_corr_main_with_rag",
    caption=r"""\textbf{What predicts essay quality (with retrieval).} Spearman rank
correlation $\rho$ between a model's mean essay \textbf{Score} and each
predictor, taken \emph{across} the $n{=}21$ models (one point per model).
\textbf{Legal Ref.\ Sim.}: overlap of statutory citations with the reference
solution; \textbf{Article Recitation}: verbatim statute-recall accuracy on a
separate task. Brackets are 95\% bootstrap CIs. Both correlate with quality, but
Legal Ref.\ Sim.\ is computed from the same essays \textbf{Score} grades---so
their agreement is partly built in---whereas Article Recitation is measured on an
independent task and is thus the stronger evidence.""",
)

In [ ]:



# EXTENDED: + tokens (in k) + cost
extended = essay_evaluation.make_summary(
    model_study_df,
    show_judge_scores=False,
    show_answer_tokens=True,
    show_judge_tokens=True,
    show_gen_cost=True,
    show_judge_cost=True,
    token_divisor=1000, token_digits=1,
    caption=r"\textbf{Essay writing performance (with law retrieval, extended).} Setup as in Table~\ref{tab:model_comparison_with_rag}: for each of the 81 GPBam exam cases the five best-matching German statute passages are retrieved (rewritten queries, hybrid search over the \emph{gesetze-im-internet} corpus) and prepended to the prompt, the model writes a full legal essay, and a three-judge LLM ensemble scores it 0.0--1.0 against the reference solution. \textbf{Score} is the mean over the 81 cases of the per-essay median across judges, as a percentage with bootstrap standard error ($_{\pm\mathrm{SE}}$); the three following columns report each individual judge's own mean score in the same way (the FP8 and API variants of the Qwen judge are one logical judge and are merged). \textbf{Legal Ref. Sim.} is the Jaccard overlap of statutory citations with the reference solution. \textbf{Answer Tokens (k)} counts only the tokens the model generated per essay -- the retrieved statute context is prompt input and is not included -- in thousands, as mean $\pm$ standard deviation across the 81 cases; \textbf{Judge Tokens (k)} is the same for the tokens generated by the full judge panel per essay. \textbf{Gen.\ Cost} and \textbf{Judge Cost} are total spend in USD over all 81 essays, for retrieval-augmented answer generation and for the judge panel respectively; blank cells mean the provider reported no cost. Models are ordered by Score (ascending).",
    label="tab:model_comparison_with_rag_extended"
)

extended

In [ ]:
res_ext = essay_correlation.correlate(
    extended,
    recitation_csv="zubaers_result/article_recitation/article_recitation.csv",
)

In [ ]:
essay_correlation.to_latex(
    res_ext,
    label="tab:essay_corr_ext_with_rag",
    caption=r"""\textbf{What predicts essay quality (extended, with retrieval).}
Spearman $\rho$ of mean essay \textbf{Score} against every other per-model
metric, across the $n{=}21$ models; brackets are 95\% bootstrap CIs. The
individual judges (\textbf{gpt-5-nano}, \textbf{DeepSeek-V4-Flash},
\textbf{qwen3.6-35b-a3b}) correlate near-perfectly by construction---\textbf{Score}
is their median. Among knowledge signals, \textbf{Article Recitation} is measured
on an independent task and is the stronger evidence, while \textbf{Legal Ref.\ Sim.}
is computed from the same essays \textbf{Score} grades, so its agreement is partly
built in; length and cost (\textbf{Answer/Judge Tokens}, \textbf{Gen./Judge Cost})
correlate only weakly. $n<21$ where cost was unreported. Rows ordered by
$|\rho|$.""",
)


In [ ]:
importlib.reload(essay_plots)
plot_result = essay_plots.make_plots(

    extended,

    show_stats_box=True,
    stats_box_location="lower right",


    exclude_models={
        "mistral-small-3.1-24b-instruct",
        "EuroLLM-22B-Instruct-2512",
        "Phi-4-mini-instruct",
        "gemma-4-31B-it-FP8-block",
    },

    exclude_from_cost_plot={
        "Qwen3.6-35B-A3B-FP8",
        "qwen3.6-35b-a3b",
        "hello"
    },

    label_models=True,
    avoid_label_overlap=True,
    use_log_scale_for_cost=False,
    remove_nonpositive_costs=True,

    save_figures=False,
    output_directory=(
        "/root/work/p25-plexam/experiments/figures/essay_quality"
    ),
    output_format="pdf",

    individual_figsize=(9, 7),
    combined_figsize=(19, 6.5),
    figure_dpi=200,
    save_dpi=300,

    x_limits=X_LIMITS,
    y_limits=Y_LIMITS,

    make_individual_plots=True,
    make_combined_plot=True,
    show_plots=True,

    combined_title=(
        "Essay quality relationships by model — with retrieval"
    ),
)

In [ ]:
importlib.reload(article_recitation_plots)
recitation_plot = article_recitation_plots.make_plot(
    extended,
    recitation_csv=(
        "zubaers_result/article_recitation/article_recitation.csv"
    ),
    title="Essay quality vs. legal knowledge — with law retrieval",
    filename="essay_vs_article_recitation_lawrag",

    save_figure=False,
    show_plot=True,

    label_models=True,
    avoid_label_overlap=True,

    show_legend=True,
    legend_location="lower right",

    show_stats_box=False,
    stats_box_location="lower left",

    x_limits=(0, 54),
    y_limits=Y_LIMITS,
)

### Judge Analysis

In [ ]:
judge_analysis_df = essay_evaluation.make_summary(
    model_study_df,
    show_judge_scores=True,
    show_legal_ref_sim=False,
    show_answer_tokens=False,
    show_judge_tokens=False,
    show_gen_cost=False,
    show_judge_cost=False,
    token_divisor=1000, token_digits=1,
    caption=r"\textbf{Comparison of ensemble and individual judge scores (with law retrieval).} Each model writes a legal essay for all 81 GPBam cases with the five best-matching retrieved statute passages prepended to the prompt. The three LLM judges evaluate each essay against the reference solution on a 0.0--1.0 scale. \textbf{Score} is the mean across cases of the per-essay median of the three judge scores, whereas the remaining columns report each judge's mean score across cases. Values are scaled to percentages and shown with bootstrap standard errors ($_{\pm\mathrm{SE}}$). The FP8 and API variants of the Qwen judge are merged into one logical judge. Differences in judges' scoring levels motivate using the median as a robust ensemble aggregation. Models are ordered by Score (ascending); higher is better.",
    label="tab:judge_analysis_with_rag"
)

judge_analysis_df

In [ ]:
methods = ["spearman", "pearson", "qwk"]


# No exclude= here: the four models dropped in the no-RAG notebook
# (mistral-small-3.1-24b-instruct, EuroLLM-22B-Instruct-2512, Phi-4-mini-instruct,
# gemma-4-31B-it-FP8-block) were never run with retrieval, so this CSV already
# contains exactly the 26 models. Passing them would raise KeyError.
agreement = judge_correlation.analyze_levels(
    model_study_df,
    methods=methods,
    n_boot=10_000,
    n_permutations=10_000,
    seed=42,
)

model_analysis = agreement["model"].copy()
essay_analysis = agreement["essay"].copy()
agreement_table = agreement["table"].copy()

display(
    agreement_table.round({
        "coefficient": 3,
        "ci_low": 3,
        "ci_high": 3,
        "p_value": 4,
    })
)

In [ ]:
table_latex = judge_correlation.agreement_table_to_latex(
    agreement["table"],
    label="tab:judge_agreement_with_rag",
    caption=(
        r"\textbf{Inter-judge agreement in the GPBam essay-writing experiment "
        r"(with law retrieval).} Agreement is evaluated pairwise for three LLM "
        r"judges. At the model level, one observation is a generation model and "
        r"each judge's value is its mean score over that model's available "
        r"essays; at the essay level, one observation is an individual essay for "
        r"which both judges supplied a score. Spearman's $\rho$ measures rank "
        r"agreement, Pearson's $r$ linear association, and quadratic-weighted "
        r"kappa (QWK, $\kappa_w$) agreement in absolute score levels. Scores lie "
        r"on 0--1 and are represented on an equivalent fixed 0--100 ordinal grid "
        r"for QWK. Entries are coefficients with percentile 95\% bootstrap "
        r"confidence intervals. Model-level intervals resample models; "
        r"essay-level intervals resample complete generation-model clusters. "
        r"Model-level $p$-values are analytic for Spearman/Pearson and "
        r"permutation-based for QWK; essay-level $p$-values are omitted because "
        r"essays from the same model are dependent. $n$ is the pairwise-complete "
        r"number of models or essays, so it varies when a judge score is missing: "
        r"the gpt-5-nano judge returned no parsable score for 58 of the 2106 "
        r"essays and the Qwen judge for 2, while all 26 models retain a "
        r"model-level mean. The FP8 and API Qwen judge columns are coalesced into "
        r"one logical judge. Higher coefficients indicate stronger agreement; QWK "
        r"equals 1 under perfect agreement."
    ),
)

In [ ]:
fig, axes = judge_correlation.plot_level_agreement(
    agreement,
    methods=["spearman", "pearson", "qwk"],
    title="Inter-judge agreement at model and essay levels — with law retrieval",
    save_path="/root/work/p25-plexam/experiments/zubaers_result/essay_writing/with_rag/ji2/judge-analysis/judge_agreement_lawrag.pdf",
    dpi=300,
)

In [ ]:
figure_latex = judge_correlation.agreement_figure_to_latex(
    "judge_agreement_lawrag.pdf",
    label="fig:judge_agreement_with_rag",
    caption=(
        r"\textbf{Inter-judge agreement at model and essay levels in the GPBam "
        r"essay-writing experiment (with law retrieval).} Points show Spearman's "
        r"$\rho$, Pearson's $r$, and quadratic-weighted kappa "
        r"(QWK, $\kappa_w$) for each judge pair; horizontal bars are percentile "
        r"95\% bootstrap confidence intervals. In the left panel, each observation "
        r"is a generation model and each judge's value is its mean over that "
        r"model's available essays, with models resampled for the intervals. In "
        r"the right panel, each observation is an individual essay scored by both "
        r"judges, with complete generation-model clusters resampled to preserve "
        r"within-model dependence. Spearman measures rank agreement and Pearson "
        r"linear association, whereas QWK additionally penalises systematic "
        r"differences in absolute score levels after representing the original "
        r"0--1 scores on an equivalent fixed 0--100 ordinal grid. The displayed "
        r"essay counts vary because correlations use pairwise-complete judge "
        r"scores and the gpt-5-nano judge returned no parsable score for 58 of "
        r"the 2106 essays. The FP8 and API Qwen columns are coalesced into one "
        r"logical judge. Higher values indicate stronger agreement."
    ),
)

In [ ]:
###
### Heatmaps of correlation matrices for model-level agreement
###

for method in ["spearman", "pearson", "qwk"]:
    fig, ax = judge_correlation.plot_correlation_heatmap(
        model_analysis["correlation_matrices"][method],
        method=method,          # title text only — see the warning below
        level="model",          # title text only
        figsize=(7, 5),
    )
    # fig.savefig(f"judge_heatmap_{method}_model.pdf", bbox_inches="tight")


In [ ]:
###
### Heatmaps of correlation matrices for essay-level agreement
###

for method in ["spearman", "pearson", "qwk"]:
    fig, ax = judge_correlation.plot_correlation_heatmap(
        essay_analysis["correlation_matrices"][method],
        method=method,          # title text only — see the warning below
        level="model",          # title text only
        figsize=(7, 5),
    )
    # fig.savefig(f"judge_heatmap_{method}_model.pdf", bbox_inches="tight")


In [ ]:
for level, analysis in [("model", model_analysis), ("essay", essay_analysis)]:
    fig, ax = judge_correlation.plot_score_distributions(
        analysis["scores"],
        level=level,            # sets the y-axis label; must match the analysis
        figsize=(9, 5),
    )
    fig.savefig(f"/root/work/p25-plexam/experiments/zubaers_result/essay_writing/with_rag/ji2/judge-analysis/judge_score_distributions_{level}.pdf", bbox_inches="tight")


# Archive

In [ ]:
# Doing the median score and average legal reference similarity score for each model in a simplified way

# Find all judge-score columns
score_cols = [
    col for col in model_study_df.columns
    if "score_Judge" in col
]

# Median judge score for each row
df_with_scores = model_study_df.assign(
    median_judge_score=model_study_df[score_cols].median(
        axis=1,
        skipna=True
    )
)

# Average judge score and average legal-reference similarity per model
model_summary = (
    df_with_scores
    .groupby("model", as_index=False)
    .agg(
        average_score=("median_judge_score", "mean"),
        average_legal_ref_sim=("legal_ref_sim", "mean")
    )
)

# Convert both columns to percentages and round to one decimal place
percentage_cols = [
    "average_score",
    "average_legal_ref_sim"
]

model_summary[percentage_cols] = (
    model_summary[percentage_cols] * 100
).round(1)

# Sort by average score
model_summary = (
    model_summary
    .sort_values("average_score", ascending=True)
    .reset_index(drop=True)
)

model_summary

In [ ]:
model_study_df[
    [col for col in model_study_df.columns if "score_Judge" in col]
    + ["legal_ref_sim", "model"]
]